# Modeliranje 1-D porazdelitve: razpadi Higgsovega bozona

## 1. naloga: Praktikum strojnega učenja v fiziki 2025/26

**predavatelj**: [prof. dr. Borut Paul Kerševan](mailto:borut.kersevan@ijs.si)  
**asistent**: [Jan Gavranovič](mailto:jan.gavranovic@ijs.si)

# Scikit-learn in Gaussovski procesi

- Pri nalogi bomo uporabili implementacijo GPR iz knjižnice scikit-learn.
- Obstajajo tudi druge:
    - [GPyTorch](https://github.com/jwangjie/gpytorch) (PyTorch),
    - [GPflow](https://github.com/GPflow/GPflow) (Tensorflow),
    - [GPJax](https://github.com/JaxGaussianProcesses/GPJax) (JAX),
    - [GPy](https://github.com/SheffieldML/GPy) (Python)

- Za zgled si bomo ogledali modeliranje točk iz $f(x) = x \sin (x)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import mplhep as mh

mh.style.use("ATLAS")

## Toy podatki

In [ ]:
n = 1000 # number of training and testing points

X = np.linspace(0.0, 10.0, n).reshape(-1, 1) # column vector 
y = np.squeeze(X * np.sin(X)) # points from true function

- Podatke razdelimo na učno in testno množico.
- Za "učenje" bomo uporabili 1% vseh točk.
- Vrednostim bomo dodali še Gaussovski šum z amplitudo $\sigma$.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99, shuffle=True)

# add some random Gaussian noise to real data y
noise_std = 0.75
y_train_noisy = y_train + np.random.normal(loc=0.0, scale=noise_std, size=y_train.shape)

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(X, y, lw=3, label="True function", zorder=0)
plt.scatter(X_train, y_train_noisy, c="C1", label="Train data", marker="x", s=100, zorder=1)

plt.title("True function and train data", fontsize=16)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()

plt.tight_layout()

## Jedra (kernels)

- Če želimo napovedati vrednosti $f$ v novi točki $\vec{x}_*$, moramo primerjati $\vec{x}_*$ z vsemi $N$ vrednostmi $\{\vec{x}_n\}$.
- Zanima nas kako podobne so si vrednosti $\vec{x}_*$ in $\vec{x}_n$.
- To podobnost izračuna jedrna funkcija $k(\vec{x}_n, \vec{x}_*) \geq 0$.
- Za $N$ podatkovni točk definiramo $N \times N$ podobnostno matriko (*kernel matrix*):
$$
K = 
\begin{bmatrix}
    k(\vec{x}_1, \vec{x}_1) & k(\vec{x}_1, \vec{x}_2) & \cdots & k(\vec{x}_1, \vec{x}_N) \\
    k(\vec{x}_2, \vec{x}_1) & k(\vec{x}_2, \vec{x}_2) & \cdots & k(\vec{x}_2, \vec{x}_N) \\
    \vdots          & \vdots          & \ddots & \vdots          \\
    k(\vec{x}_N, \vec{x}_1) & k(\vec{x}_N, \vec{x}_2) & \cdots & k(\vec{x}_N, \vec{x}_N)
\end{bmatrix} \>.
$$

- Jedro RBF je poznano tudi pod imenom jedro SE (*squared exponential*), brez parametrov je:
$$
k(\vec{x}_i, \vec{x}_j^\prime) = \exp \left( (\vec{x}_i - \vec{x}_j)^\top (\vec{x}_i - \vec{x}_j) \right) \>.
$$
- Najbol enostavno in največkrat uporabljeno

In [ ]:
def se_kernel_explicit(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Squared exponential kernel computed explicitly."""
    n = x.shape[0]  # number of points in x
    m = y.shape[0]  # number of points in y
    d = x.shape[1]  # dimensionality

    # initialize the kernel matrix
    K = np.zeros((n, m))

    # compute kernel value for each pair of points: O(n x m x d)
    for i in range(n):
        for j in range(m):
            # compute squared Euclidean distance between x[i] and y[j]
            squared_distance = 0.0
            for k in range(d): # in 1D this is simply d=1
                diff = x[i, k] - y[j, k]
                squared_distance += diff ** 2

            # apply the squared exponential kernel formula
            K[i, j] = np.exp(-0.5 * squared_distance)

    return K

def se_kernel(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    "Squared exponential kernel: vectorized."
    x_norm = np.sum(x**2, axis=1)
    y_norm = np.sum(y**2, axis=1)
    sq_dist = x_norm.reshape(-1, 1) + y_norm - 2 * np.dot(x, y.T)
    return np.exp(-0.5 * sq_dist)

In [ ]:
K = se_kernel(X, X) # n x n kernel matrix

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(K)
plt.colorbar()

plt.xticks([])
plt.yticks([])

plt.title("RBF kernel matrix", fontsize=20)
plt.xlabel(r"$x$", fontsize=20)
plt.ylabel(r"$x^\prime$", fontsize=20)

plt.tight_layout()

## Gaussovski proces

- Gaussovski proces je porazdelitev po funkcijah:
$$
f(\vec{x}) \sim \text{GP}(m(\vec{x}), k(\vec{x}, \vec{x}^\prime)) \>.
$$
- Povprečje $m$ in varianca $k$ sta funkciji vhodnih podatkov $\vec{x}$.
- Glavne ideje:
   - Neparametrična metoda regresije, ki nam poleg napovedi pove tudi napake teh napovedi.
   - Predpostavka je, da so podatki porazdeljeni Gaussovsko.
   - Kernel nam pove, da so si točke blizu podobne.

- Uporabili bomo knjižnjico sci-kit learn, ki implementira GPR (algoritem je opisan v `gpr_teorija.ipynb`)
- Definiramo jedro in GPR model.
- Parameter $\alpha$ interpretiramo kot varianco Gaussovskega šuma vhodnih podatkov (merske napake!).

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

In [ ]:
kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
gaussian_process = GaussianProcessRegressor(kernel=kernel, alpha=noise_std**2, n_restarts_optimizer=10)

- Na modelu kličemo `fit` funkcijo na učnih podatkih.

In [ ]:
gaussian_process.fit(X_train, y_train_noisy)

- Model optimizira tudi parametra $\sigma^2$ in $\ell$ jedra RBF.

In [ ]:
gaussian_process.kernel_

- Za napovedovanje novih vrednosti na testnih podatkih uporabimo `predict`.

In [ ]:
mean_prediction, std_prediction = gaussian_process.predict(X_test, return_std=True)

In [ ]:
# flatten for plotting
X_train, X_test = X_train.flatten(), X_test.flatten()
mean_prediction, std_prediction = mean_prediction.flatten(), std_prediction.flatten()

In [ ]:
# sort for plotting
idx = np.argsort(X_test)
X_test, mean_prediction, std_prediction = X_test[idx], mean_prediction[idx], std_prediction[idx]

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(X, y, "b-", label="true function")
plt.errorbar(
    X_train,
    y_train_noisy,
    noise_std,
    linestyle="None",
    color="k",
    marker=".",
    markersize=12,
    label="train",
    capsize=3,
)
plt.plot(X_test, mean_prediction, label="mean prediction", c="r")
plt.fill_between(
    X_test,
    mean_prediction - 1.96 * std_prediction,
    mean_prediction + 1.96 * std_prediction,
    color="gray",
    alpha=0.2,
    label="95% confidence interval",
)

plt.title("GPR prediction", fontsize=20)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()

plt.tight_layout()

## Iskanje optimalnih parametrov jedra

- Poiskali bomo najboljša parametra $\sigma^2$ in $\ell$ našega jedra RBF.
- Izračunali bomo $\log p(\mathbf{y} |X, \theta)$ pri različnih vrednostih teh parametrov in poiskali maksimum.
- Bolj učinkovit način bi bil z uporabe kakšne od gradientnih metod.

In [ ]:
# make a log grid search space
length_scale = np.logspace(-3, 3, 100)
noise_level = np.logspace(-3, 3, 100)

- Log marginal likelihodd je za GPR model:
$$
\log p(\mathbf{y} | X, \theta) = \underbrace{-\frac{1}{2} \mathbf{y}^T (K_\theta + \sigma^2 I)^{-1} \mathbf{y}}_{\text{data fit}} - \underbrace{\frac{1}{2} \log |K_\theta + \sigma^2 I|}_{\text{complexity penalty}} - \underbrace{\frac{n}{2} \log(2\pi)}_{\text{constant}}
$$

In [ ]:
# do grid scan and calculate mll at each point
mll_results = np.empty((len(length_scale), len(noise_level)))

for i, scale in enumerate(length_scale):
    for j, noise in enumerate(noise_level):
        mll_results[i, j] = - gaussian_process.log_marginal_likelihood(theta=np.log([scale, noise]))

In [ ]:
# find the smallest element in mll
idx = np.unravel_index(np.argmin(mll_results), mll_results.shape)
best_noise_level, best_length_scale = length_scale[idx[0]], noise_level[idx[1]]

best_noise_level, best_length_scale

In [ ]:
mh.style.use()
plt.figure(figsize=(7, 6))

plt.pcolormesh(length_scale, noise_level, mll_results)
plt.colorbar()

plt.scatter(best_length_scale, best_noise_level, marker="x", s=150, c="r")

plt.yscale("log")
plt.xscale("log")

plt.xlabel("length scale", fontsize=16)
plt.ylabel("noise level", fontsize=16)
plt.title("log marginal likelihood", fontsize=16)

plt.tight_layout()

## Uporaba različnih jeder

- V scikit-learn je na voljo več različnih jeder.
- Jedra lahko med sabo tudi kombiniramo:
    - $k(\vec{x}, \vec{x}^\prime) = c k_1(\vec{x}, \vec{x}^\prime)$ za konstanto $c>0$,
    - $k(\vec{x}, \vec{x}^\prime) = f(\vec{x})k_1(\vec{x}, \vec{x}^\prime)f(\vec{x}^\prime)$ za neko funkcijo $f$,
    - $k(\vec{x}, \vec{x}^\prime) = q(k_1(\vec{x}, \vec{x}^\prime))$ za polinom $q$ s pozitivnimi koeficienti,
    - $k(\vec{x}, \vec{x}^\prime) = \exp(k_1(\vec{x}, \vec{x}^\prime))$,
    - $k(\vec{x}, \vec{x}^\prime) = \vec{x}^\top A \vec{x}^\prime$ za pozitivno semi deifnitno matriko $A$,
    - $k(\vec{x}, \vec{x}^\prime) = k_1(\vec{x}, \vec{x}^\prime) + k_2(\vec{x}, \vec{x}^\prime)$,
    - $k(\vec{x}, \vec{x}^\prime) = k_1(\vec{x}, \vec{x}^\prime) \cdot k_2(\vec{x}, \vec{x}^\prime)$.
- Glej https://www.cs.toronto.edu/~duvenaud/cookbook/. 

In [ ]:
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic, ExpSineSquared, ConstantKernel, DotProduct, Matern

In [ ]:
f_model = lambda x: np.sin((x - 2.5)**2)

np.random.seed(12)

X_train = np.random.uniform(0, 5, 10).reshape(-1, 1)
y_train = f_model(X_train[:, 0])

X_test = np.linspace(0, 5, 100).reshape(-1, 1)
y_true = f_model(X_test[:, 0])

In [ ]:
kernels = {
    "RBF": 1.0 * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-1, 10.0),
    ),
    "RationalQuadratic": 1.0 * RationalQuadratic(
        length_scale=1.0,
        alpha=0.1,
        alpha_bounds=(1e-5, 1e15),
    ),
    "ExpSineSquared": 1.0 * ExpSineSquared(
        length_scale=1.0,
        periodicity=3.0,
        length_scale_bounds=(0.1, 10.0),
        periodicity_bounds=(1.0, 10.0),
    ),
    "DotProduct": ConstantKernel(
        constant_value=0.1,
        constant_value_bounds=(0.01, 10.0),
    )
    * DotProduct(
        sigma_0=1.0,
        sigma_0_bounds=(0.1, 10.0),
    ) ** 2,
    "Matern": 1.0 * Matern(
        length_scale=1.0,
        length_scale_bounds=(1e-1, 10.0),
        nu=1.5,
    ),
}

In [ ]:
fig, axs = plt.subplots(len(kernels), figsize=(9, 5 * len(kernels)))

for i, (name, kernel) in enumerate(kernels.items()):
    gpr_model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
    gpr_model.fit(X_train, y_train)

    y_mean, y_std = gpr_model.predict(X_test, return_std=True)

    axs[i].scatter(X_train[:, 0], y_train, c="r", label="train data", zorder=3)
    axs[i].plot(X_test[:, 0], y_true, c="b", ls="--", alpha=0.8, label="true function", zorder=2)
    axs[i].plot(X_test[:, 0], y_mean, color="k", label="predicted mean", zorder=1)
    axs[i].fill_between(
        X_test[:, 0],
        y_mean - y_std,
        y_mean + y_std,
        alpha=0.1,
        color="black",
        label=r"$\pm$ 1 std. dev.",
    )
    axs[i].set_xlabel(r"$x$", fontsize=12)
    axs[i].set_ylabel(r"$y$", fontsize=12)
    axs[i].set_title(name)
    axs[i].legend(loc="upper right")

fig.tight_layout()